In [51]:
import pandas as pd
from google.cloud import storage
import tempfile, os
from IPython.display import display

In [52]:
from google.cloud import storage

GCP_PROJECT = "moeyens-thor-dev"
storage_client = storage.Client(project=GCP_PROJECT)

/Users/b612foundation/b612_packages/adam-thor-research/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [53]:
# Local path to the downloaded analysis_observations.parquet file
analysis_observations_file = "/Users/b612foundation/b612_packages/data/analysis_observations.parquet"

In [54]:
# First, get the analysis observations we need from the local parquet file.
# This file is large, so only load the columns needed for labeling transformed detections.

#analysis_observations_df = pd.read_parquet(
#    analysis_observations_file,``
#    columns=["id", "object_id"],  # orbit id / None for noise
#)

#analysis_observations_df.head()


In [55]:
#write this to a parquet file for faster loading  in the future 
#analysis_observations_df.to_parquet("analysis_observations_small.parquet")

reload = False

if reload == True:

    analysis_observations_df = pd.read_parquet("analysis_observations_small.parquet")

In [56]:
import numpy as np
import pandas as pd

def compute_linearity(group: pd.DataFrame) -> pd.Series:
    # unpack nested coordinates into plain columns
    coords = pd.json_normalize(group["coordinates"])
    x = coords["theta_x"].to_numpy()
    y = coords["theta_y"].to_numpy()
    n = len(x)
    if n < 2:
        return pd.Series({"n_obs": n, "linearity_r2": np.nan})

    # least-squares line fit: y ≈ m x + b
    m, b = np.polyfit(x, y, 1)
    y_hat = m * x + b

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

    return pd.Series({"n_obs": n, "linearity_r2": r2})

In [ ]:
# GCS configuration
GCP_PROJECT = "moeyens-thor-dev"
BUCKET = "thor-research"
BASE_PREFIX = "runs/dec3"

# Analysis configuration
MIN_OBS_PER_ORBIT = 6  # stop when we find any object with >= 6 observations

CHECKPOINT_PATH = "/Users/b612foundation/b612_packages/data/linearity_results.parquet"
CHECKPOINT_EVERY = 10

storage_client = storage.Client(project=GCP_PROJECT)
bucket = storage_client.bucket(BUCKET)

# Find all transformed_detections.parquet files for the test orbits
blobs = storage_client.list_blobs(BUCKET, prefix=f"{BASE_PREFIX}/")
orbit_parquet_paths = sorted(
    [blob.name for blob in blobs if blob.name.endswith("transformed_detections.parquet")]
)

print(f"Found {len(orbit_parquet_paths)} candidate orbit files.")

recovered_observations_df = None
orbits_with_enough_obs = None
selected_orbit_path = None

all_summaries = []
i = 0

for blob_name in orbit_parquet_paths:
    i += 1
    print(f"Processing {i} of {len(orbit_parquet_paths)}")
    gcs_uri = f"gs://{BUCKET}/{blob_name}"
    print(f"Checking {gcs_uri} ...")

    # Download this orbit's transformed detections to a temporary file and load with pandas
    blob = bucket.blob(blob_name)
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
        blob.download_to_filename(tmp.name)
        tmp_path = tmp.name
    
    # Full test-orbit slug, e.g. "40085_r1.349_e0.000_nu105.000_psi-15.000"
    test_orbit_slug = blob_name.split("/")[-2]
    test_orbit_id = test_orbit_slug.split("_")[0]
    print(f"Test orbit ID: {test_orbit_id}")

    df = pd.read_parquet(tmp_path)
    os.remove(tmp_path)

    #print(df.head())

    print(f"DF before merging: {df.shape}")

    df_with_labels = df.merge(analysis_observations_df, on="id", how="left")

    #print(df_with_labels.head())

    #now remove the none in the object_id column 
    print(f"DF size before filtering noise: {df_with_labels.shape}")
    df_with_labels = df_with_labels[df_with_labels['object_id'].notna()]
    print(f"DF size after filtering noise: {df_with_labels.shape}")

    #now group by object_id and filter for the ones with more than 6 observations 
    grouped_by_orbit = df_with_labels.groupby("object_id")

    for object_id, count in grouped_by_orbit["object_id"].value_counts().items():
        print(f"Object ID before filtering: {object_id}, Count: {count}")
    #print(grouped_by_orbit.head())
    orbits_with_enough_obs = grouped_by_orbit.filter(lambda g: len(g) >= MIN_OBS_PER_ORBIT)

    #print size of enough 
    print(f"DF size after filtering for 6+ observations: {orbits_with_enough_obs.shape}")

    #print each object_id in orbits_with_enough_obs and number of observations it has 
    for object_id, count in orbits_with_enough_obs["object_id"].value_counts().items():
        print(f"Object ID: {object_id}, Count: {count}")

    summary = (
        orbits_with_enough_obs
        .groupby("object_id")
        .apply(compute_linearity)
        .reset_index()
    )
    print(summary)
    print(blob_name)

    summary["test_orbit_id"] = test_orbit_id
    summary["test_orbit_slug"] = test_orbit_slug
    summary = summary[["test_orbit_id", "test_orbit_slug", "object_id", "n_obs", "linearity_r2"]]

    all_summaries.append(summary)

    if len(all_summaries) % CHECKPOINT_EVERY == 0:
        all_results = pd.concat(all_summaries, ignore_index=True)
        all_results.to_parquet(CHECKPOINT_PATH, index=False)  # overwrites file
    

/Users/b612foundation/b612_packages/adam-thor-research/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Found 5145 candidate orbit files.
Processing 1 of 5145
Checking gs://thor-research/runs/dec3/link_test_orbit_40085_r1.349_e0.000_nu105.000_psi-15.000/outputs/40085_r1.349_e0.000_nu105.000_psi-15.000/transformed_detections.parquet ...
Test orbit ID: 40085
DF before merging: (41319, 4)
DF size before filtering noise: (41319, 5)
DF size after filtering noise: (75, 5)
Object ID before filtering: 1277587, Count: 1
Object ID before filtering: 1288956, Count: 5
Object ID before filtering: 1606956, Count: 2
Object ID before filtering: 2154794, Count: 1
Object ID before filtering: 2558755, Count: 2
Object ID before filtering: 2637035, Count: 1
Object ID before filtering: 271668, Count: 4
Object ID before filtering: 2777030, Count: 6
Object ID before filtering: 2804312, Count: 1
Object ID before filtering: 3081310, Count: 1
Object ID before filtering: 4028756, Count: 1
Object ID before filtering: 4356813, Count: 1
Object ID before filtering: 4528040, Count: 1
Object ID before filtering: 4655679,

/var/folders/jn/6gvx57b56bjdzktff3dt1p3r0000gn/T/ipykernel_70195/4156983323.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_linearity)


Test orbit ID: 40085
DF before merging: (45658, 4)
DF size before filtering noise: (45658, 5)
DF size after filtering noise: (49, 5)
Object ID before filtering: 1277587, Count: 1
Object ID before filtering: 1288956, Count: 1
Object ID before filtering: 1795975, Count: 1
Object ID before filtering: 2131346, Count: 4
Object ID before filtering: 2154794, Count: 1
Object ID before filtering: 271668, Count: 4
Object ID before filtering: 2777030, Count: 4
Object ID before filtering: 2864236, Count: 1
Object ID before filtering: 2980411, Count: 1
Object ID before filtering: 3524721, Count: 1
Object ID before filtering: 4028756, Count: 2
Object ID before filtering: 4356813, Count: 1
Object ID before filtering: 4528040, Count: 1
Object ID before filtering: 4655679, Count: 2
Object ID before filtering: 4762333, Count: 1
Object ID before filtering: 5362952, Count: 1
Object ID before filtering: 5505338, Count: 3
Object ID before filtering: 5596856, Count: 4
Object ID before filtering: 5783183, Cou

/var/folders/jn/6gvx57b56bjdzktff3dt1p3r0000gn/T/ipykernel_70195/4156983323.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_linearity)


Test orbit ID: 40085
DF before merging: (47736, 4)
DF size before filtering noise: (47736, 5)
DF size after filtering noise: (45, 5)
Object ID before filtering: 1043871, Count: 1
Object ID before filtering: 1277587, Count: 1
Object ID before filtering: 1352281, Count: 1
Object ID before filtering: 1554517, Count: 1
Object ID before filtering: 2131346, Count: 5
Object ID before filtering: 2154794, Count: 1
Object ID before filtering: 2161530, Count: 1
Object ID before filtering: 2265097, Count: 2
Object ID before filtering: 2626422, Count: 1
Object ID before filtering: 271668, Count: 2
Object ID before filtering: 2761450, Count: 1
Object ID before filtering: 2777030, Count: 2
Object ID before filtering: 2864236, Count: 1
Object ID before filtering: 2980411, Count: 1
Object ID before filtering: 3524721, Count: 1
Object ID before filtering: 4028756, Count: 1
Object ID before filtering: 4356813, Count: 1
Object ID before filtering: 4364785, Count: 7
Object ID before filtering: 4528040, Cou

/var/folders/jn/6gvx57b56bjdzktff3dt1p3r0000gn/T/ipykernel_70195/4156983323.py:81: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_linearity)


Test orbit ID: 40085
DF before merging: (47264, 4)


In [ ]:
#for each test orbit, group observations with the same orbit id  
# keep the objects with more than 6 observations 

MIN_OBS_PER_ORBIT = 6  

# Keep only orbits with more than 6 observations
orbits_with_enough_obs = (
    recovered_observations_df
    .groupby("orbit_id")
    .filter(lambda g: len(g) >= MIN_OBS_PER_ORBIT)
)

print(
    f"{orbits_with_enough_obs['orbit_id'].nunique()} orbits with ≥{MIN_OBS_PER_ORBIT} observations "
    f"out of {recovered_observations_df['orbit_id'].nunique()} total orbits."
)

# Optional: get a dict from orbit_id -> DataFrame of its observations
orbits_by_id = {
    orbit_id: df for orbit_id, df in orbits_with_enough_obs.groupby("orbit_id")
}

In [ ]:
# for each of these, determine how linear the observations are 